# LLM auto-grading -- Kaggle runner

Clones the public code repo ([https://github.com/ptran1203/msc.git](https://github.com/ptran1203/msc.git)) and attaches the private data
via the Kaggle dataset `ptran1203/llm-autograding-course-data` (must be added under Notebook ->
Add Input before running). Course rules forbid distributing the real student data
outside the class, so it lives only in that private dataset, never in the repo itself.

Edit the **config cell** below, then Run All.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
%cd /kaggle/working
!rm -rf msc
!git clone --depth 1 https://github.com/ptran1203/msc.git
%cd /kaggle/working/msc/llm
!pip install -q -r requirements.txt


In [ ]:
import shutil, pathlib
dataset_root = pathlib.Path("/kaggle/input/llm-autograding-course-data")
repo_llm = pathlib.Path("/kaggle/working/msc/llm")
shutil.copytree(dataset_root / "data", repo_llm / "data")
(repo_llm / "synth").mkdir(exist_ok=True)
shutil.copytree(dataset_root / "synth_generated", repo_llm / "synth" / "generated")
print("data staged OK")


## Run -- generates the public submission (328 samples, all 3 tasks, static few-shot)

To try a different config, just edit the command line directly:
`--split sample|public`, `--tasks 1,2,3`, `--fewshot` / `--rag` / neither (zero-shot).

In [ ]:
%cd /kaggle/working/msc/llm/src
!python -u run_prompting.py --split public --tasks 1,2,3 --fewshot


In [ ]:
import glob, re
for path in sorted(glob.glob("../outputs/predictions_task*_public_*.json")):
    task_num = re.search(r"predictions_task(\d)_", path).group(1)
    get_ipython().system(f"python validate_submission.py {task_num} {path} --split public")


## Output

Predictions and logs are written to `/kaggle/working/msc/llm/outputs/` -- anything under
`/kaggle/working` is saved as this notebook's output automatically on commit. The cell
above validates each generated `predictions_task*.json` against the schema before you
trust it as submission-ready.